## Importaciones y configuración inicial

In [120]:
# Importaciones generales
from pathlib import Path
import pandas as pd
import numpy as np
import shutil

# Dataset de Hugging Face
from datasets import load_from_disk, DatasetDict, Audio

# Visualización
import matplotlib.pyplot as plt

# Audio
from IPython.display import Audio as IPythonAudio, display
import librosa

# Utilidades
import random
from datetime import datetime

In [22]:
# Configuración general
SEED = 42

# Rutas principales del proyecto
RUTA_DATASET_ORIGINAL = Path(r"C:\Lara\datasets\lara_whisper_dataset")
RUTA_DATASET_PREPARADO = Path(r"C:\Lara\datasets\lara_whisper_dataset_preparado")

RUTA_RESULTADOS_CUADERNO_03 = Path(r"C:\Lara\resultados\03_mejora_modelo")
RUTA_RESULTADOS_CUADERNO_04 = Path(r"C:\Lara\resultados\04_analisis_errores_modelo_mejorado")

RUTA_SALIDA_CUADERNO_05 = Path(r"C:\Lara\resultados\05_limpieza_dataset")
RUTA_DATASET_LIMPIO_V2 = Path(r"C:\Lara\datasets\lara_whisper_dataset_limpio_v2")

RUTA_BASE_AUDIOS = Path(r"C:\Lara\audios-230426\audios-s3")

# Creamos la carpeta de salida del cuaderno 05 si no existe
RUTA_SALIDA_CUADERNO_05.mkdir(parents=True, exist_ok=True)

print("Ruta dataset original:", RUTA_DATASET_ORIGINAL)
print("Ruta dataset preparado:", RUTA_DATASET_PREPARADO)
print("Ruta resultados cuaderno 03:", RUTA_RESULTADOS_CUADERNO_03)
print("Ruta resultados cuaderno 04:", RUTA_RESULTADOS_CUADERNO_04)
print("Ruta salida cuaderno 05:", RUTA_SALIDA_CUADERNO_05)
print("Ruta dataset limpio v2:", RUTA_DATASET_LIMPIO_V2)
print("Ruta base audios:", RUTA_BASE_AUDIOS)

Ruta dataset original: C:\Lara\datasets\lara_whisper_dataset
Ruta dataset preparado: C:\Lara\datasets\lara_whisper_dataset_preparado
Ruta resultados cuaderno 03: C:\Lara\resultados\03_mejora_modelo
Ruta resultados cuaderno 04: C:\Lara\resultados\04_analisis_errores_modelo_mejorado
Ruta salida cuaderno 05: C:\Lara\resultados\05_limpieza_dataset
Ruta dataset limpio v2: C:\Lara\datasets\lara_whisper_dataset_limpio_v2
Ruta base audios: C:\Lara\audios-230426\audios-s3


## Carga del dataset original

In [3]:
# Cargamos el dataset original desde disco
dataset_original = load_from_disk(str(RUTA_DATASET_ORIGINAL))

print("Dataset original cargado correctamente")
print(dataset_original)

Dataset original cargado correctamente
Dataset({
    features: ['audio', 'texto'],
    num_rows: 40842
})


In [4]:
# Añadimos un índice original antes de dividir el dataset
dataset_original = dataset_original.add_column(
    "indice_original",
    list(range(len(dataset_original)))
)

# Dividimos primero el dataset original en train y temporal
split_inicial = dataset_original.train_test_split(
    test_size=0.20,
    seed=SEED
)

dataset_train = split_inicial["train"]
dataset_temporal = split_inicial["test"]

# Dividimos el 20% temporal en test y eval
# El 80% de este bloque será test y el 20% será eval
split_temporal = dataset_temporal.train_test_split(
    test_size=0.20,
    seed=SEED
)

dataset_test = split_temporal["train"]
dataset_eval = split_temporal["test"]

print("División del dataset completada")
print("----------------------------------------")
print("Train:", len(dataset_train))
print("Test: ", len(dataset_test))
print("Eval: ", len(dataset_eval))
print("----------------------------------------")
print("Total:", len(dataset_train) + len(dataset_test) + len(dataset_eval))

print("----------------------------------------")
print("Columnas train:", dataset_train.column_names)
print("Columnas test: ", dataset_test.column_names)
print("Columnas eval: ", dataset_eval.column_names)

División del dataset completada
----------------------------------------
Train: 32673
Test:  6535
Eval:  1634
----------------------------------------
Total: 40842
----------------------------------------
Columnas train: ['audio', 'texto', 'indice_original']
Columnas test:  ['audio', 'texto', 'indice_original']
Columnas eval:  ['audio', 'texto', 'indice_original']


## Carga de resultados del cuaderno 04

In [5]:
# Listamos los archivos generados en el cuaderno 04
archivos_cuaderno_04 = sorted(RUTA_RESULTADOS_CUADERNO_04.glob("*"))

print("Archivos encontrados en resultados del cuaderno 04:")
print("----------------------------------------")

for archivo in archivos_cuaderno_04:
    if archivo.is_file():
        print("Archivo:", archivo.name)
    elif archivo.is_dir():
        print("Carpeta:", archivo.name)

Archivos encontrados en resultados del cuaderno 04:
----------------------------------------
Archivo: aciertos_iniciales_empeorados.csv
Archivo: alucinaciones_largas_modelo_inicial.csv
Archivo: alucinaciones_largas_modelo_mejorado.csv
Archivo: casos_mayores_empeoramientos.csv
Archivo: casos_mayores_mejoras.csv
Archivo: comparativa_con_revision_manual.csv
Archivo: comparativa_modelo_inicial_vs_mejorado.csv
Archivo: errores_corregidos_completamente.csv
Archivo: errores_graves_modelo_mejorado.csv
Archivo: resumen_alucinaciones_largas.csv
Archivo: resumen_error_por_longitud_frase.csv
Archivo: resumen_metricas_modelo_inicial_vs_mejorado.csv
Archivo: resumen_metricas_sin_posibles_problemas_dataset.csv
Archivo: resumen_muestras_mejora_empeora_igual.csv
Archivo: resumen_revision_manual.csv
Archivo: revision_manual_posibles_desajustes_audio_texto.csv


In [6]:
# Cargamos los archivos principales generados en el cuaderno 04
ruta_comparativa = RUTA_RESULTADOS_CUADERNO_04 / "comparativa_modelo_inicial_vs_mejorado.csv"
ruta_revision_manual = RUTA_RESULTADOS_CUADERNO_04 / "revision_manual_posibles_desajustes_audio_texto.csv"
ruta_comparativa_revision = RUTA_RESULTADOS_CUADERNO_04 / "comparativa_con_revision_manual.csv"

df_comparativa = pd.read_csv(ruta_comparativa)
df_revision_manual = pd.read_csv(ruta_revision_manual)
df_comparativa_revision = pd.read_csv(ruta_comparativa_revision)

print("Archivos cargados correctamente")
print("----------------------------------------")
print("Comparativa modelo inicial vs mejorado:", df_comparativa.shape)
print("Revisión manual:", df_revision_manual.shape)
print("Comparativa con revisión manual:", df_comparativa_revision.shape)

Archivos cargados correctamente
----------------------------------------
Comparativa modelo inicial vs mejorado: (6535, 10)
Revisión manual: (29, 10)
Comparativa con revisión manual: (6535, 20)


In [7]:
# Revisamos las columnas disponibles en cada dataframe
print("Columnas df_comparativa:")
print(df_comparativa.columns.tolist())

print("\nColumnas df_revision_manual:")
print(df_revision_manual.columns.tolist())

print("\nColumnas df_comparativa_revision:")
print(df_comparativa_revision.columns.tolist())

Columnas df_comparativa:
['indice_test', 'texto_real', 'prediccion_modelo_inicial', 'prediccion_modelo_mejorado', 'wer_inicial', 'wer_mejorado', 'cer_inicial', 'cer_mejorado', 'diferencia_wer', 'diferencia_cer']

Columnas df_revision_manual:
['indice_test', 'texto_real_dataset', 'prediccion_modelo_inicial', 'prediccion_modelo_mejorado', 'observacion_manual', 'posible_problema_dataset', 'wer_inicial', 'wer_mejorado', 'cer_inicial', 'cer_mejorado']

Columnas df_comparativa_revision:
['indice_test', 'texto_real', 'prediccion_modelo_inicial', 'prediccion_modelo_mejorado', 'wer_inicial', 'wer_mejorado', 'cer_inicial', 'cer_mejorado', 'diferencia_wer', 'diferencia_cer', 'resultado_comparacion', 'num_palabras_texto_real', 'longitud_frase', 'num_palabras_pred_inicial', 'num_palabras_pred_mejorado', 'exceso_palabras_inicial', 'exceso_palabras_mejorado', 'revisado_manualmente', 'posible_problema_dataset', 'observacion_manual']


## Cruce de resultados con el índice original del dataset

In [8]:
# Evitamos que Hugging Face intente decodificar los audios al recorrer el dataset
dataset_test_sin_decode = dataset_test.cast_column("audio", Audio(decode=False))

# Creamos una tabla auxiliar para relacionar el índice del test con el índice original
registros_test = []

for indice_test in range(len(dataset_test_sin_decode)):
    fila = dataset_test_sin_decode[indice_test]

    registros_test.append({
        "indice_test": indice_test,
        "indice_original": fila["indice_original"],
        "texto_dataset_test": fila["texto"],
        "ruta_audio": fila["audio"]["path"]
    })

df_indices_test = pd.DataFrame(registros_test)

print("Tabla de índices creada correctamente")
print("----------------------------------------")
print(df_indices_test.shape)

df_indices_test.head()

Tabla de índices creada correctamente
----------------------------------------
(6535, 4)


,indice_test,indice_original,texto_dataset_test,ruta_audio
0,0,29890,EN LA LATA HAY LIMONADA.,662819171ef22d020bf25236_1744267180.wav
1,1,16715,ME ASOMARÉ DESDE EL DÉCIMO PISO.,662819171ef22d020bf25236_1732174228.wav
2,2,29236,AFRONTAR LA AFRENTA DE ALFREDO.,662819171ef22d020bf25236_1743662803.wav
3,3,6120,HOY TENGO QUE COGER EL BUS PARA IR A LA BIBLIO...,663a416baa128a933ae405e4_1715095223.wav
4,4,36641,EL CIELO ES CALIDO,6915ab1cadc93acb24a300a0_1768820872.wav


In [9]:
# Cruzamos los resultados del cuaderno 04 con el índice original del dataset
df_limpieza_base = df_comparativa_revision.merge(
    df_indices_test,
    on="indice_test",
    how="left"
)

print("DataFrame base para limpieza creado")
print("----------------------------------------")
print("Filas:", len(df_limpieza_base))
print("Columnas:", len(df_limpieza_base.columns))

print("\nColumnas principales:")
print([
    "indice_test",
    "indice_original",
    "texto_real",
    "texto_dataset_test",
    "posible_problema_dataset",
    "observacion_manual"
])

df_limpieza_base[
    [
        "indice_test",
        "indice_original",
        "texto_real",
        "texto_dataset_test",
        "posible_problema_dataset",
        "observacion_manual"
    ]
].head()

DataFrame base para limpieza creado
----------------------------------------
Filas: 6535
Columnas: 23

Columnas principales:
['indice_test', 'indice_original', 'texto_real', 'texto_dataset_test', 'posible_problema_dataset', 'observacion_manual']


,indice_test,indice_original,texto_real,texto_dataset_test,posible_problema_dataset,observacion_manual
0,0,29890,EN LA LATA HAY LIMONADA.,EN LA LATA HAY LIMONADA.,False,NaN
1,1,16715,ME ASOMARÉ DESDE EL DÉCIMO PISO.,ME ASOMARÉ DESDE EL DÉCIMO PISO.,False,NaN
2,2,29236,AFRONTAR LA AFRENTA DE ALFREDO.,AFRONTAR LA AFRENTA DE ALFREDO.,False,NaN
3,3,6120,HOY TENGO QUE COGER EL BUS PARA IR A LA BIBLIO...,HOY TENGO QUE COGER EL BUS PARA IR A LA BIBLIO...,False,NaN
4,4,36641,EL CIELO ES CALIDO,EL CIELO ES CALIDO,False,NaN


In [10]:
# Comprobamos si hay registros sin índice original después del cruce
num_indices_nulos = df_limpieza_base["indice_original"].isna().sum()

# Comprobamos si el texto del CSV coincide con el texto recuperado del dataset_test
textos_coinciden = (
    df_limpieza_base["texto_real"].astype(str).str.strip()
    ==
    df_limpieza_base["texto_dataset_test"].astype(str).str.strip()
)

num_textos_no_coinciden = (~textos_coinciden).sum()

print("Validación del cruce")
print("----------------------------------------")
print("Registros sin indice_original:", num_indices_nulos)
print("Registros con texto no coincidente:", num_textos_no_coinciden)
print("Total registros:", len(df_limpieza_base))

Validación del cruce
----------------------------------------
Registros sin indice_original: 0
Registros con texto no coincidente: 0
Total registros: 6535


## Identificación de muestras problemáticas

In [11]:
# Revisamos cuántas muestras tienen revisión manual y cuántas están marcadas como problema
resumen_revision = df_limpieza_base["revisado_manualmente"].value_counts(dropna=False)
resumen_problema = df_limpieza_base["posible_problema_dataset"].value_counts(dropna=False)

print("Resumen de revisión manual")
print("----------------------------------------")
print(resumen_revision)

print("\nResumen de posible problema de dataset")
print("----------------------------------------")
print(resumen_problema)

Resumen de revisión manual
----------------------------------------
revisado_manualmente
False    6506
True       29
Name: count, dtype: int64

Resumen de posible problema de dataset
----------------------------------------
posible_problema_dataset
False    6525
True       10
Name: count, dtype: int64


In [12]:
# Extraemos las muestras que han sido marcadas manualmente como posibles problemas del dataset
df_problemas_manuales = df_limpieza_base[
    df_limpieza_base["posible_problema_dataset"] == True
].copy()

print("Muestras marcadas manualmente como problema de dataset:")
print(len(df_problemas_manuales))

df_problemas_manuales[
    [
        "indice_test",
        "indice_original",
        "texto_real",
        "prediccion_modelo_inicial",
        "prediccion_modelo_mejorado",
        "wer_inicial",
        "wer_mejorado",
        "cer_inicial",
        "cer_mejorado",
        "observacion_manual"
    ]
].head(10)

Muestras marcadas manualmente como problema de dataset:
10


,indice_test,indice_original,texto_real,prediccion_modelo_inicial,prediccion_modelo_mejorado,wer_inicial,wer_mejorado,cer_inicial,cer_mejorado,observacion_manual
834,834,16706,ESTUVE EN LA CIMA DE AQUEL FAMOSO PICO.,ESTUVE EN LA CIMA DE AQUEL CIMA POR LA CIMA DE...,ESTUVE EN LA CIMA DE AQUEL CIMA DE CIMA POR LA...,0.750000,1.250000,0.487179,0.820513,El texto de referencia del dataset no correspo...
1080,1080,7097,¡JUANJO ES EL JEFE CORTANDO JAMÓN!,¡JONATHAN NIEGA JUEGA CON EL JARDÍN HUMO EL JA...,¡JUANITO ES UN JARDÍN! QUE TE VA A ENTRAR DIFE...,1.333333,2.666667,1.000000,2.176471,El texto de referencia del dataset no correspo...
1339,1339,3193,NO SE OYE BIEN,"VICENTE AL MONTAR DE LAS BOTELLAS DE CINE, SE ...",ME LLAMÓ YESICA A LA MI HERMANA IR A RESTAURAN...,3.750000,7.250000,5.000000,9.357143,El texto real del dataset no parece correspond...
1506,1506,16688,ME DAN MUCHA RISA LOS PAYASOS.,SALTÉ A LA COMIDA CON MI PIENSA MUCHO ELADO.,SALTÉ A LA COMIDA TANTO EN LA COMIDA TULA CON ...,1.500000,2.666667,1.100000,1.800000,El texto de referencia del dataset no correspo...
2897,2897,34653,LA URRACA SE METIÓ EN LA PERRERA.,UNA CARTATATA SE MIRA ENTRÓ POR LA CARRE.,UNA CARTA QUE ME HACE ABANÉS DE SER MUY EMOTIV...,0.857143,2.285714,0.636364,1.606061,El texto de referencia del dataset no correspo...
3491,3491,7482,CLODOMIRO JUEGA CON CLOTILDE AL CLUEDO.,COMEREMOS CROQUETAS CON CREMA.,"EL COLOR DEL MAR Y DEL CIELO ES AZUL, EL COLOR...",0.833333,2.333333,0.692308,1.230769,El texto de referencia del dataset no correspo...
3756,3756,21707,"LA BOTELLA LLEVA AGUA CON LIMÓN, SE LLAMA AQUA...","LA BOTELLA LLUEVE CON MI LÁPIZ,VERO Y PAYE A L...",LA BOTELLA LLUEVE CON LA LLUEVE EN LA AVINGUDA...,1.000000,1.666667,0.588235,1.098039,El texto de referencia del dataset no correspo...
4392,4392,31081,GREGORIO COME EN EL RESTAURANTE GRANAINO.,CLODOMIRO COME EN EL TÚNEL DE CENA COTILANDO C...,¡CORREROS! QUE CANCIÓN MÁS COCHE QUE SE COMPRÓ...,1.500000,2.333333,0.951220,1.487805,El texto de referencia del dataset no correspo...
5252,5252,238,La escribiente llevaba medias de licra.,MARÍA ES UN PILOTO MUY PORTUGUÍA.,MEA ENCANTA IR A MERCADOS LOCALES Y PROBAR DIF...,1.000000,2.333333,0.923077,2.000000,El texto de referencia del dataset no correspo...
5695,5695,37435,"PERDONE, ¿DÓNDE PARA LA ESTACIÓN DE AUTOBUSES?",PEDURA ME MONTÓ EN SU PEDALEANDO PEDRA LE GUST...,PERDUCTOR DE CONSUMIR TRADUCTOR PARA PROTEGER ...,1.714286,1.857143,0.869565,1.586957,El texto de referencia del dataset no correspo...


In [13]:
# Creamos columnas de decisión para documentar el motivo de limpieza
df_limpieza_base["accion_limpieza"] = "conservar"
df_limpieza_base["motivo_limpieza"] = ""

# Las muestras marcadas manualmente como problema se excluirán del dataset limpio
mascara_problema_manual = df_limpieza_base["posible_problema_dataset"] == True

df_limpieza_base.loc[mascara_problema_manual, "accion_limpieza"] = "excluir"
df_limpieza_base.loc[mascara_problema_manual, "motivo_limpieza"] = "marcado_manual_posible_problema_dataset"

print("Resumen de acciones de limpieza")
print("----------------------------------------")
print(df_limpieza_base["accion_limpieza"].value_counts())

print("\nMotivos de limpieza")
print("----------------------------------------")
print(df_limpieza_base["motivo_limpieza"].value_counts())

Resumen de acciones de limpieza
----------------------------------------
accion_limpieza
conservar    6525
excluir        10
Name: count, dtype: int64

Motivos de limpieza
----------------------------------------
motivo_limpieza
                                           6525
marcado_manual_posible_problema_dataset      10
Name: count, dtype: int64


In [14]:
# Revisamos las muestras que por ahora quedarían excluidas
df_excluir_manual = df_limpieza_base[
    df_limpieza_base["accion_limpieza"] == "excluir"
].copy()

print("Muestras excluidas por revisión manual:", len(df_excluir_manual))

df_excluir_manual[
    [
        "indice_test",
        "indice_original",
        "texto_real",
        "prediccion_modelo_mejorado",
        "wer_mejorado",
        "cer_mejorado",
        "observacion_manual",
        "motivo_limpieza"
    ]
]

Muestras excluidas por revisión manual: 10


,indice_test,indice_original,texto_real,prediccion_modelo_mejorado,wer_mejorado,cer_mejorado,observacion_manual,motivo_limpieza
834,834,16706,ESTUVE EN LA CIMA DE AQUEL FAMOSO PICO.,ESTUVE EN LA CIMA DE AQUEL CIMA DE CIMA POR LA...,1.250000,0.820513,El texto de referencia del dataset no correspo...,marcado_manual_posible_problema_dataset
1080,1080,7097,¡JUANJO ES EL JEFE CORTANDO JAMÓN!,¡JUANITO ES UN JARDÍN! QUE TE VA A ENTRAR DIFE...,2.666667,2.176471,El texto de referencia del dataset no correspo...,marcado_manual_posible_problema_dataset
1339,1339,3193,NO SE OYE BIEN,ME LLAMÓ YESICA A LA MI HERMANA IR A RESTAURAN...,7.250000,9.357143,El texto real del dataset no parece correspond...,marcado_manual_posible_problema_dataset
1506,1506,16688,ME DAN MUCHA RISA LOS PAYASOS.,SALTÉ A LA COMIDA TANTO EN LA COMIDA TULA CON ...,2.666667,1.800000,El texto de referencia del dataset no correspo...,marcado_manual_posible_problema_dataset
2897,2897,34653,LA URRACA SE METIÓ EN LA PERRERA.,UNA CARTA QUE ME HACE ABANÉS DE SER MUY EMOTIV...,2.285714,1.606061,El texto de referencia del dataset no correspo...,marcado_manual_posible_problema_dataset
3491,3491,7482,CLODOMIRO JUEGA CON CLOTILDE AL CLUEDO.,"EL COLOR DEL MAR Y DEL CIELO ES AZUL, EL COLOR...",2.333333,1.230769,El texto de referencia del dataset no correspo...,marcado_manual_posible_problema_dataset
3756,3756,21707,"LA BOTELLA LLEVA AGUA CON LIMÓN, SE LLAMA AQUA...",LA BOTELLA LLUEVE CON LA LLUEVE EN LA AVINGUDA...,1.666667,1.098039,El texto de referencia del dataset no correspo...,marcado_manual_posible_problema_dataset
4392,4392,31081,GREGORIO COME EN EL RESTAURANTE GRANAINO.,¡CORREROS! QUE CANCIÓN MÁS COCHE QUE SE COMPRÓ...,2.333333,1.487805,El texto de referencia del dataset no correspo...,marcado_manual_posible_problema_dataset
5252,5252,238,La escribiente llevaba medias de licra.,MEA ENCANTA IR A MERCADOS LOCALES Y PROBAR DIF...,2.333333,2.000000,El texto de referencia del dataset no correspo...,marcado_manual_posible_problema_dataset
5695,5695,37435,"PERDONE, ¿DÓNDE PARA LA ESTACIÓN DE AUTOBUSES?",PERDUCTOR DE CONSUMIR TRADUCTOR PARA PROTEGER ...,1.857143,1.586957,El texto de referencia del dataset no correspo...,marcado_manual_posible_problema_dataset


In [15]:
# Creamos columnas para marcar candidatos automáticos a revisión
df_limpieza_base["candidato_revision_automatica"] = False
df_limpieza_base["motivo_revision_automatica"] = ""

# Regla 1: error muy alto en el modelo mejorado
mascara_error_muy_alto = (
    (df_limpieza_base["wer_mejorado"] >= 2.0) |
    (df_limpieza_base["cer_mejorado"] >= 1.0)
)

# Regla 2: predicción mejorada con exceso importante de palabras
mascara_exceso_palabras = df_limpieza_base["exceso_palabras_mejorado"] >= 5

# Regla 3: empeoramiento fuerte respecto al modelo inicial
mascara_empeoramiento_fuerte = (
    (df_limpieza_base["diferencia_wer"] >= 1.0) |
    (df_limpieza_base["diferencia_cer"] >= 0.5)
)

# Combinamos las reglas automáticas
mascara_candidato_automatico = (
    mascara_error_muy_alto |
    mascara_exceso_palabras |
    mascara_empeoramiento_fuerte
)

df_limpieza_base.loc[
    mascara_candidato_automatico,
    "candidato_revision_automatica"
] = True

# Guardamos el motivo de forma acumulativa
df_limpieza_base.loc[mascara_error_muy_alto, "motivo_revision_automatica"] += "error_muy_alto;"
df_limpieza_base.loc[mascara_exceso_palabras, "motivo_revision_automatica"] += "exceso_palabras;"
df_limpieza_base.loc[mascara_empeoramiento_fuerte, "motivo_revision_automatica"] += "empeoramiento_fuerte;"

print("Candidatos automáticos a revisión:")
print(df_limpieza_base["candidato_revision_automatica"].value_counts())

print("\nMotivos detectados:")
print(df_limpieza_base["motivo_revision_automatica"].value_counts().head(20))

Candidatos automáticos a revisión:
candidato_revision_automatica
False    5204
True     1331
Name: count, dtype: int64

Motivos detectados:
motivo_revision_automatica
                                                        5204
empeoramiento_fuerte;                                    706
error_muy_alto;                                          392
error_muy_alto;exceso_palabras;                          176
exceso_palabras;                                          29
error_muy_alto;empeoramiento_fuerte;                      24
error_muy_alto;exceso_palabras;empeoramiento_fuerte;       4
Name: count, dtype: int64


In [16]:
# Candidatos automáticos que todavía no están excluidos por revisión manual
df_candidatos_revision = df_limpieza_base[
    (df_limpieza_base["candidato_revision_automatica"] == True) &
    (df_limpieza_base["accion_limpieza"] != "excluir")
].copy()

print("Candidatos automáticos pendientes de revisión manual:", len(df_candidatos_revision))

df_candidatos_revision[
    [
        "indice_test",
        "indice_original",
        "texto_real",
        "prediccion_modelo_mejorado",
        "wer_mejorado",
        "cer_mejorado",
        "exceso_palabras_mejorado",
        "diferencia_wer",
        "diferencia_cer",
        "motivo_revision_automatica"
    ]
].sort_values(
    by=["cer_mejorado", "wer_mejorado"],
    ascending=False
).head(20)

Candidatos automáticos pendientes de revisión manual: 1321


,indice_test,indice_original,texto_real,prediccion_modelo_mejorado,wer_mejorado,cer_mejorado,exceso_palabras_mejorado,diferencia_wer,diferencia_cer,motivo_revision_automatica
2001,2001,40667,Hola,O HAGAS FUEGO JUNTO AL LAGO.,6.000000,6.750000,5,1.000000,0.500000,error_muy_alto;exceso_palabras;empeoramiento_f...
2389,2389,38626,COLOCAME,"CORRER Y GANAR, TODO ES EMPEZAR.",6.000000,3.375000,5,-1.000000,-1.000000,error_muy_alto;exceso_palabras;
5843,5843,32697,QUIERO AGUA,¡CRISTINA Y CRISTIAN COLECCIONAN CROMOS!,2.500000,3.272727,3,0.500000,-1.363636,error_muy_alto;
1590,1590,37554,ROSA SE RÍE,LOLA SE ACLARA EL AUTOBUSES ESTÁ MUY SUBIDO,2.333333,3.181818,5,-0.666667,-1.454545,error_muy_alto;exceso_palabras;
3365,3365,36175,ROSA SE RÍE,LOLA SE MIRA LLEVÓ EL LASTE EL CUENTO.,2.333333,2.727273,5,0.333333,0.000000,error_muy_alto;exceso_palabras;
5976,5976,25810,EL BEBE BUCEA,EL FONTANERO ARREGLÓ LA FUERA DEL FUERIÓN.,2.000000,2.615385,4,-1.000000,-1.307692,error_muy_alto;
3128,3128,11005,JUAN DOBLÓ EL FOLIO BLANCO.,¡JUANJO ES EL JEFE CORTANDO SALTANDO TANTAS TO...,2.800000,2.555556,10,-1.600000,-1.814815,error_muy_alto;exceso_palabras;
4604,4604,39709,ESTE ALCOHOL HUELE MAL.,EL EQUIPO DE ATLETISMO DE MI ESCUELA GANÓ EL C...,3.250000,2.521739,9,-1.750000,-1.260870,error_muy_alto;exceso_palabras;
1416,1416,16408,FRAN ES FEO,CON EL FRANCÉS SUBIÓ A LA FRAGATA.,2.333333,2.454545,4,-0.666667,-0.545455,error_muy_alto;
574,574,7922,¿PUEDO COSER EN TU TELAR?,"CON EL CRITERIO DE ESTA REYES, ME CRUDO QUE EL...",3.200000,2.440000,11,-2.000000,-1.520000,error_muy_alto;exceso_palabras;


## Priorización de candidatos a revisión

In [17]:
# Clasificamos los candidatos automáticos por nivel de sospecha
df_limpieza_base["nivel_sospecha"] = "sin_sospecha"

# Sospecha baja: cualquier regla automática
df_limpieza_base.loc[
    df_limpieza_base["candidato_revision_automatica"] == True,
    "nivel_sospecha"
] = "baja"

# Sospecha media: errores altos o exceso notable
mascara_sospecha_media = (
    (df_limpieza_base["cer_mejorado"] >= 1.5) |
    (df_limpieza_base["wer_mejorado"] >= 2.5) |
    (df_limpieza_base["exceso_palabras_mejorado"] >= 6)
)

df_limpieza_base.loc[
    mascara_sospecha_media,
    "nivel_sospecha"
] = "media"

# Sospecha alta: errores muy extremos o alucinaciones largas
mascara_sospecha_alta = (
    (df_limpieza_base["cer_mejorado"] >= 2.0) |
    (df_limpieza_base["wer_mejorado"] >= 3.0) |
    (df_limpieza_base["exceso_palabras_mejorado"] >= 8)
)

df_limpieza_base.loc[
    mascara_sospecha_alta,
    "nivel_sospecha"
] = "alta"

print("Resumen por nivel de sospecha")
print("----------------------------------------")
print(df_limpieza_base["nivel_sospecha"].value_counts())

Resumen por nivel de sospecha
----------------------------------------
nivel_sospecha
sin_sospecha    5204
baja            1168
media            110
alta              53
Name: count, dtype: int64


In [18]:
# Seleccionamos solo los casos de sospecha alta que todavía no están excluidos
df_revision_prioritaria = df_limpieza_base[
    (df_limpieza_base["nivel_sospecha"] == "alta") &
    (df_limpieza_base["accion_limpieza"] != "excluir")
].copy()

# Ordenamos por los casos más graves
df_revision_prioritaria = df_revision_prioritaria.sort_values(
    by=["cer_mejorado", "wer_mejorado", "exceso_palabras_mejorado"],
    ascending=False
)

print("Casos de revisión prioritaria:", len(df_revision_prioritaria))

df_revision_prioritaria[
    [
        "indice_test",
        "indice_original",
        "texto_real",
        "prediccion_modelo_mejorado",
        "wer_mejorado",
        "cer_mejorado",
        "exceso_palabras_mejorado",
        "motivo_revision_automatica"
    ]
].head(30)

Casos de revisión prioritaria: 43


,indice_test,indice_original,texto_real,prediccion_modelo_mejorado,wer_mejorado,cer_mejorado,exceso_palabras_mejorado,motivo_revision_automatica
2001,2001,40667,Hola,O HAGAS FUEGO JUNTO AL LAGO.,6.000000,6.750000,5,error_muy_alto;exceso_palabras;empeoramiento_f...
2389,2389,38626,COLOCAME,"CORRER Y GANAR, TODO ES EMPEZAR.",6.000000,3.375000,5,error_muy_alto;exceso_palabras;
5843,5843,32697,QUIERO AGUA,¡CRISTINA Y CRISTIAN COLECCIONAN CROMOS!,2.500000,3.272727,3,error_muy_alto;
1590,1590,37554,ROSA SE RÍE,LOLA SE ACLARA EL AUTOBUSES ESTÁ MUY SUBIDO,2.333333,3.181818,5,error_muy_alto;exceso_palabras;
3365,3365,36175,ROSA SE RÍE,LOLA SE MIRA LLEVÓ EL LASTE EL CUENTO.,2.333333,2.727273,5,error_muy_alto;exceso_palabras;
5976,5976,25810,EL BEBE BUCEA,EL FONTANERO ARREGLÓ LA FUERA DEL FUERIÓN.,2.000000,2.615385,4,error_muy_alto;
3128,3128,11005,JUAN DOBLÓ EL FOLIO BLANCO.,¡JUANJO ES EL JEFE CORTANDO SALTANDO TANTAS TO...,2.800000,2.555556,10,error_muy_alto;exceso_palabras;
4604,4604,39709,ESTE ALCOHOL HUELE MAL.,EL EQUIPO DE ATLETISMO DE MI ESCUELA GANÓ EL C...,3.250000,2.521739,9,error_muy_alto;exceso_palabras;
1416,1416,16408,FRAN ES FEO,CON EL FRANCÉS SUBIÓ A LA FRAGATA.,2.333333,2.454545,4,error_muy_alto;
574,574,7922,¿PUEDO COSER EN TU TELAR?,"CON EL CRITERIO DE ESTA REYES, ME CRUDO QUE EL...",3.200000,2.440000,11,error_muy_alto;exceso_palabras;


In [19]:
# Preparamos un dataframe específico para revisión manual prioritaria
df_revision_prioritaria_manual = df_revision_prioritaria[
    [
        "indice_test",
        "indice_original",
        "ruta_audio",
        "texto_real",
        "prediccion_modelo_inicial",
        "prediccion_modelo_mejorado",
        "wer_inicial",
        "wer_mejorado",
        "cer_inicial",
        "cer_mejorado",
        "exceso_palabras_mejorado",
        "motivo_revision_automatica"
    ]
].copy()

# Añadimos columnas para rellenar manualmente
df_revision_prioritaria_manual["revisado_manualmente"] = False
df_revision_prioritaria_manual["posible_problema_dataset"] = False
df_revision_prioritaria_manual["observacion_manual"] = ""

# Guardamos el CSV para revisión manual
ruta_revision_prioritaria = RUTA_SALIDA_CUADERNO_05 / "revision_prioritaria_sospecha_alta.csv"

df_revision_prioritaria_manual.to_csv(
    ruta_revision_prioritaria,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo de revisión prioritaria guardado en:")
print(ruta_revision_prioritaria)

print("\nNúmero de casos incluidos:", len(df_revision_prioritaria_manual))

Archivo de revisión prioritaria guardado en:
C:\Lara\resultados\05_limpieza_dataset\revision_prioritaria_sospecha_alta.csv

Número de casos incluidos: 43


In [23]:
# Función para escuchar una muestra concreta de la revisión prioritaria
def reproducir_revision_prioritaria(posicion):
    fila = df_revision_prioritaria_manual.iloc[posicion]

    ruta_audio = Path(fila["ruta_audio"])

    # Si la ruta guardada es solo el nombre del archivo, la completamos con la ruta base de audios
    if not ruta_audio.is_absolute():
        ruta_audio = RUTA_BASE_AUDIOS / ruta_audio

    print("Posición en revisión prioritaria:", posicion)
    print("Índice test:", fila["indice_test"])
    print("Índice original:", fila["indice_original"])
    print("----------------------------------------")
    print("Texto real:")
    print(fila["texto_real"])
    print("----------------------------------------")
    print("Predicción modelo inicial:")
    print(fila["prediccion_modelo_inicial"])
    print("----------------------------------------")
    print("Predicción modelo mejorado:")
    print(fila["prediccion_modelo_mejorado"])
    print("----------------------------------------")
    print("WER mejorado:", round(fila["wer_mejorado"], 4))
    print("CER mejorado:", round(fila["cer_mejorado"], 4))
    print("Exceso palabras mejorado:", fila["exceso_palabras_mejorado"])
    print("Motivo revisión automática:", fila["motivo_revision_automatica"])
    print("----------------------------------------")
    print("Ruta audio:")
    print(ruta_audio)

    if not ruta_audio.exists():
        print("----------------------------------------")
        print("No se ha encontrado el archivo de audio.")
        return

    # Cargamos el audio con librosa para evitar problemas de decodificación directa
    audio_array, sampling_rate = librosa.load(str(ruta_audio), sr=16000)

    display(IPythonAudio(audio_array, rate=sampling_rate))

In [25]:
# Función para marcar una muestra de revisión prioritaria como revisada
def marcar_revision_prioritaria(posicion, posible_problema_dataset, observacion_manual=""):
    global df_revision_prioritaria_manual

    if posicion < 0 or posicion >= len(df_revision_prioritaria_manual):
        print("Posición fuera de rango.")
        return

    # Marcamos la revisión manual
    df_revision_prioritaria_manual.loc[
        df_revision_prioritaria_manual.index[posicion],
        "revisado_manualmente"
    ] = True

    df_revision_prioritaria_manual.loc[
        df_revision_prioritaria_manual.index[posicion],
        "posible_problema_dataset"
    ] = bool(posible_problema_dataset)

    df_revision_prioritaria_manual.loc[
        df_revision_prioritaria_manual.index[posicion],
        "observacion_manual"
    ] = observacion_manual

    # Guardamos el CSV actualizado
    df_revision_prioritaria_manual.to_csv(
        ruta_revision_prioritaria,
        index=False,
        encoding="utf-8-sig"
    )

    print("Revisión guardada correctamente")
    print("----------------------------------------")
    print("Posición:", posicion)
    print("Índice test:", df_revision_prioritaria_manual.iloc[posicion]["indice_test"])
    print("Índice original:", df_revision_prioritaria_manual.iloc[posicion]["indice_original"])
    print("Posible problema dataset:", posible_problema_dataset)
    print("Observación:", observacion_manual)

In [115]:
# Escuchamos un caso de revisión prioritaria
reproducir_revision_prioritaria(42)

Posición en revisión prioritaria: 42
Índice test: 4219
Índice original: 33213
----------------------------------------
Texto real:
LA ECONOMÍA DE ELCHE SE FRACTURA FRECUENTEMENTE.
----------------------------------------
Predicción modelo inicial:
LAS PÁGINAS DE ESTE LIBRO SON DE PAPEL VEGETAL.
----------------------------------------
Predicción modelo mejorado:
LA FAMILIA DE FERNANDO COME FABADA EL FIN DE SEMANA Y LA FINAL DE SEMANA.
----------------------------------------
WER mejorado: 1.8571
CER mejorado: 1.0208
Exceso palabras mejorado: 8
Motivo revisión automática: error_muy_alto;exceso_palabras;
----------------------------------------
Ruta audio:
C:\Lara\audios-230426\audios-s3\662819171ef22d020bf25236_1749105073.wav


C:\Users\abelg\AppData\Local\Temp\ipykernel_10248\30779360.py:38: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(str(ruta_audio), sr=16000)
c:\Users\abelg\AppData\Local\Programs\Python\Python313\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [103]:
marcar_revision_prioritaria(
    posicion=37,
    posible_problema_dataset=True,
    observacion_manual="El audio no corresponde con el texto de referencia."
)

Revisión guardada correctamente
----------------------------------------
Posición: 37
Índice test: 3357
Índice original: 22532
Posible problema dataset: True
Observación: El audio no corresponde con el texto de referencia.


In [113]:
marcar_revision_prioritaria(
    posicion=42,
    posible_problema_dataset=False,
    observacion_manual="El audio corresponde con el texto. Error del modelo, no del dataset."
)

Revisión guardada correctamente
----------------------------------------
Posición: 42
Índice test: 4219
Índice original: 33213
Posible problema dataset: False
Observación: El audio corresponde con el texto. Error del modelo, no del dataset.


In [116]:
# Recargamos el CSV de revisión prioritaria por si se ha actualizado desde Excel u otra herramienta
df_revision_prioritaria_manual = pd.read_csv(ruta_revision_prioritaria)

print("Revisión prioritaria cargada")
print("----------------------------------------")
print("Total casos:", len(df_revision_prioritaria_manual))
print("Revisados:", df_revision_prioritaria_manual["revisado_manualmente"].sum())
print("Pendientes:", (~df_revision_prioritaria_manual["revisado_manualmente"]).sum())

print("\nResumen posible problema dataset:")
print(df_revision_prioritaria_manual["posible_problema_dataset"].value_counts(dropna=False))

Revisión prioritaria cargada
----------------------------------------
Total casos: 43
Revisados: 43
Pendientes: 0

Resumen posible problema dataset:
posible_problema_dataset
False    30
True     13
Name: count, dtype: int64


In [117]:
# Seleccionamos solo las muestras revisadas manualmente dentro de la revisión prioritaria
df_revision_prioritaria_revisada = df_revision_prioritaria_manual[
    df_revision_prioritaria_manual["revisado_manualmente"] == True
].copy()

# Separamos las muestras confirmadas como problema
indices_originales_problema_prioritario = df_revision_prioritaria_revisada[
    df_revision_prioritaria_revisada["posible_problema_dataset"] == True
]["indice_original"].tolist()

print("Muestras revisadas en revisión prioritaria:", len(df_revision_prioritaria_revisada))
print("Muestras confirmadas como problema:", len(indices_originales_problema_prioritario))

# Marcamos como excluir las muestras confirmadas como problema
mascara_problema_prioritario = df_limpieza_base["indice_original"].isin(indices_originales_problema_prioritario)

df_limpieza_base.loc[mascara_problema_prioritario, "accion_limpieza"] = "excluir"

df_limpieza_base.loc[
    mascara_problema_prioritario,
    "motivo_limpieza"
] = df_limpieza_base.loc[
    mascara_problema_prioritario,
    "motivo_limpieza"
].replace("", "revision_prioritaria_confirmada_problema_dataset")

# Si ya tenía un motivo previo, añadimos el nuevo motivo sin perder el anterior
mascara_con_motivo_previo = (
    mascara_problema_prioritario &
    (df_limpieza_base["motivo_limpieza"] != "revision_prioritaria_confirmada_problema_dataset")
)

df_limpieza_base.loc[
    mascara_con_motivo_previo,
    "motivo_limpieza"
] = df_limpieza_base.loc[
    mascara_con_motivo_previo,
    "motivo_limpieza"
] + ";revision_prioritaria_confirmada_problema_dataset"

print("Decisiones de limpieza actualizadas")
print("----------------------------------------")
print(df_limpieza_base["accion_limpieza"].value_counts())

print("\nMotivos de limpieza:")
print(df_limpieza_base["motivo_limpieza"].value_counts())

Muestras revisadas en revisión prioritaria: 43
Muestras confirmadas como problema: 13
Decisiones de limpieza actualizadas
----------------------------------------
accion_limpieza
conservar    6512
excluir        23
Name: count, dtype: int64

Motivos de limpieza:
motivo_limpieza
                                                    6512
revision_prioritaria_confirmada_problema_dataset      13
marcado_manual_posible_problema_dataset               10
Name: count, dtype: int64


In [118]:
# Revisamos todas las muestras que se excluirán del dataset limpio
df_muestras_a_excluir = df_limpieza_base[
    df_limpieza_base["accion_limpieza"] == "excluir"
].copy()

print("Total de muestras a excluir:", len(df_muestras_a_excluir))

df_muestras_a_excluir[
    [
        "indice_test",
        "indice_original",
        "texto_real",
        "prediccion_modelo_mejorado",
        "wer_mejorado",
        "cer_mejorado",
        "motivo_limpieza",
        "observacion_manual"
    ]
].sort_values("indice_original").head(50)

Total de muestras a excluir: 23


,indice_test,indice_original,texto_real,prediccion_modelo_mejorado,wer_mejorado,cer_mejorado,motivo_limpieza,observacion_manual
5252,5252,238,La escribiente llevaba medias de licra.,MEA ENCANTA IR A MERCADOS LOCALES Y PROBAR DIF...,2.333333,2.000000,marcado_manual_posible_problema_dataset,El texto de referencia del dataset no correspo...
1339,1339,3193,NO SE OYE BIEN,ME LLAMÓ YESICA A LA MI HERMANA IR A RESTAURAN...,7.250000,9.357143,marcado_manual_posible_problema_dataset,El texto real del dataset no parece correspond...
557,557,5152,RENATO ES RUBIO.,RENATO ES RUBIO PORTÁTIL EN MI CHELO PARA COME...,2.666667,2.437500,revision_prioritaria_confirmada_problema_dataset,NaN
1080,1080,7097,¡JUANJO ES EL JEFE CORTANDO JAMÓN!,¡JUANITO ES UN JARDÍN! QUE TE VA A ENTRAR DIFE...,2.666667,2.176471,marcado_manual_posible_problema_dataset,El texto de referencia del dataset no correspo...
3491,3491,7482,CLODOMIRO JUEGA CON CLOTILDE AL CLUEDO.,"EL COLOR DEL MAR Y DEL CIELO ES AZUL, EL COLOR...",2.333333,1.230769,marcado_manual_posible_problema_dataset,El texto de referencia del dataset no correspo...
1946,1946,10465,JUANI TRADUJO TODO.,A LALO LE GUSTA LA LECHE EN UN CHOCOLATE.,3.000000,1.736842,revision_prioritaria_confirmada_problema_dataset,NaN
6467,6467,11166,LA ORUGA TIENE OJOS,EL PRIMER JUEVES DE JUNIO NOS IREMOS DE JUERGA.,2.250000,2.000000,revision_prioritaria_confirmada_problema_dataset,NaN
1506,1506,16688,ME DAN MUCHA RISA LOS PAYASOS.,SALTÉ A LA COMIDA TANTO EN LA COMIDA TULA CON ...,2.666667,1.800000,marcado_manual_posible_problema_dataset,El texto de referencia del dataset no correspo...
834,834,16706,ESTUVE EN LA CIMA DE AQUEL FAMOSO PICO.,ESTUVE EN LA CIMA DE AQUEL CIMA DE CIMA POR LA...,1.250000,0.820513,marcado_manual_posible_problema_dataset,El texto de referencia del dataset no correspo...
3756,3756,21707,"LA BOTELLA LLEVA AGUA CON LIMÓN, SE LLAMA AQUA...",LA BOTELLA LLUEVE CON LA LLUEVE EN LA AVINGUDA...,1.666667,1.098039,marcado_manual_posible_problema_dataset,El texto de referencia del dataset no correspo...


In [121]:
# Obtenemos los índices originales de las muestras que se excluirán
indices_originales_excluir = sorted(
    df_muestras_a_excluir["indice_original"].astype(int).unique().tolist()
)

print("Número de muestras a excluir:", len(indices_originales_excluir))
print("Primeros índices a excluir:")
print(indices_originales_excluir[:20])

Número de muestras a excluir: 23
Primeros índices a excluir:
[238, 3193, 5152, 7097, 7482, 10465, 11166, 16688, 16706, 21707, 21944, 22532, 25810, 30680, 31081, 31371, 34484, 34493, 34653, 36175]


In [122]:
# Creamos la lista de índices que se conservarán
indices_excluir_set = set(indices_originales_excluir)

indices_conservar = [
    indice for indice in range(len(dataset_original))
    if indice not in indices_excluir_set
]

# Creamos el nuevo dataset limpio seleccionando solo las muestras válidas
dataset_limpio_v2 = dataset_original.select(indices_conservar)

print("Dataset limpio v2 creado")
print("----------------------------------------")
print("Registros dataset original:", len(dataset_original))
print("Registros excluidos:", len(indices_originales_excluir))
print("Registros dataset limpio v2:", len(dataset_limpio_v2))
print("----------------------------------------")
print("Comprobación total:", len(dataset_limpio_v2) + len(indices_originales_excluir))

Dataset limpio v2 creado
----------------------------------------
Registros dataset original: 40842
Registros excluidos: 23
Registros dataset limpio v2: 40819
----------------------------------------
Comprobación total: 40842


In [123]:
# Si ya existe una versión previa del dataset limpio v2, la eliminamos para poder guardar de nuevo
if RUTA_DATASET_LIMPIO_V2.exists():
    shutil.rmtree(RUTA_DATASET_LIMPIO_V2)

# Guardamos el dataset limpio en disco
dataset_limpio_v2.save_to_disk(str(RUTA_DATASET_LIMPIO_V2))

print("Dataset limpio v2 guardado correctamente en:")
print(RUTA_DATASET_LIMPIO_V2)

Saving the dataset (0/9 shards):   0%|          | 0/40819 [00:00<?, ? examples/s]

Dataset limpio v2 guardado correctamente en:
C:\Lara\datasets\lara_whisper_dataset_limpio_v2


In [124]:
# Guardamos un CSV con las muestras excluidas para documentar la limpieza aplicada
ruta_muestras_excluidas = RUTA_SALIDA_CUADERNO_05 / "muestras_excluidas_dataset_limpio_v2.csv"

df_muestras_a_excluir.to_csv(
    ruta_muestras_excluidas,
    index=False,
    encoding="utf-8-sig"
)

print("Registro de muestras excluidas guardado en:")
print(ruta_muestras_excluidas)

Registro de muestras excluidas guardado en:
C:\Lara\resultados\05_limpieza_dataset\muestras_excluidas_dataset_limpio_v2.csv


In [126]:
# Creamos un conjunto con los índices originales que deben excluirse
indices_excluir_set = set(indices_originales_excluir)

# Evitamos que Hugging Face intente decodificar los audios durante el filtrado
dataset_train_sin_decode = dataset_train.cast_column("audio", Audio(decode=False))
dataset_test_sin_decode = dataset_test.cast_column("audio", Audio(decode=False))
dataset_eval_sin_decode = dataset_eval.cast_column("audio", Audio(decode=False))

# Filtramos cada split conservando su asignación original
dataset_train_limpio_v2 = dataset_train_sin_decode.filter(
    lambda fila: fila["indice_original"] not in indices_excluir_set
)

dataset_test_limpio_v2 = dataset_test_sin_decode.filter(
    lambda fila: fila["indice_original"] not in indices_excluir_set
)

dataset_eval_limpio_v2 = dataset_eval_sin_decode.filter(
    lambda fila: fila["indice_original"] not in indices_excluir_set
)

print("Splits limpios creados")
print("----------------------------------------")
print("Train original:", len(dataset_train))
print("Train limpio:  ", len(dataset_train_limpio_v2))
print("----------------------------------------")
print("Test original: ", len(dataset_test))
print("Test limpio:   ", len(dataset_test_limpio_v2))
print("----------------------------------------")
print("Eval original: ", len(dataset_eval))
print("Eval limpio:   ", len(dataset_eval_limpio_v2))
print("----------------------------------------")
print(
    "Total limpio:",
    len(dataset_train_limpio_v2)
    + len(dataset_test_limpio_v2)
    + len(dataset_eval_limpio_v2)
)

Filter:   0%|          | 0/32673 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6535 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1634 [00:00<?, ? examples/s]

Splits limpios creados
----------------------------------------
Train original: 32673
Train limpio:   32673
----------------------------------------
Test original:  6535
Test limpio:    6512
----------------------------------------
Eval original:  1634
Eval limpio:    1634
----------------------------------------
Total limpio: 40819


In [127]:
# Creamos un DatasetDict con los splits limpios
dataset_limpio_v2_split = DatasetDict({
    "train": dataset_train_limpio_v2,
    "test": dataset_test_limpio_v2,
    "eval": dataset_eval_limpio_v2
})

# Ruta de guardado para el dataset limpio con splits
RUTA_DATASET_LIMPIO_V2_SPLIT = Path(r"C:\Lara\datasets\lara_whisper_dataset_limpio_v2_split")

# Si existe una versión previa, la eliminamos
if RUTA_DATASET_LIMPIO_V2_SPLIT.exists():
    shutil.rmtree(RUTA_DATASET_LIMPIO_V2_SPLIT)

# Guardamos el DatasetDict en disco
dataset_limpio_v2_split.save_to_disk(str(RUTA_DATASET_LIMPIO_V2_SPLIT))

print("Dataset limpio v2 con splits guardado correctamente en:")
print(RUTA_DATASET_LIMPIO_V2_SPLIT)

print("\nResumen del dataset guardado:")
print(dataset_limpio_v2_split)

Saving the dataset (0/7 shards):   0%|          | 0/32673 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/6512 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1634 [00:00<?, ? examples/s]

Dataset limpio v2 con splits guardado correctamente en:
C:\Lara\datasets\lara_whisper_dataset_limpio_v2_split

Resumen del dataset guardado:
DatasetDict({
    train: Dataset({
        features: ['audio', 'texto', 'indice_original'],
        num_rows: 32673
    })
    test: Dataset({
        features: ['audio', 'texto', 'indice_original'],
        num_rows: 6512
    })
    eval: Dataset({
        features: ['audio', 'texto', 'indice_original'],
        num_rows: 1634
    })
})


In [128]:
# Creamos un registro de los índices originales conservados en cada split
registros_indices_split = []

for nombre_split, dataset_split in dataset_limpio_v2_split.items():
    indices_originales = dataset_split["indice_original"]
    textos = dataset_split["texto"]

    for indice_original, texto in zip(indices_originales, textos):
        registros_indices_split.append({
            "split": nombre_split,
            "indice_original": indice_original,
            "texto": texto
        })

df_indices_splits_limpios = pd.DataFrame(registros_indices_split)

ruta_indices_splits_limpios = RUTA_SALIDA_CUADERNO_05 / "indices_splits_limpios_v2.csv"

df_indices_splits_limpios.to_csv(
    ruta_indices_splits_limpios,
    index=False,
    encoding="utf-8-sig"
)

print("CSV de índices de splits limpios guardado en:")
print(ruta_indices_splits_limpios)

print("\nResumen por split:")
print(df_indices_splits_limpios["split"].value_counts())

CSV de índices de splits limpios guardado en:
C:\Lara\resultados\05_limpieza_dataset\indices_splits_limpios_v2.csv

Resumen por split:
split
train    32673
test      6512
eval      1634
Name: count, dtype: int64


In [129]:
# Creamos un resumen final de la limpieza aplicada
resumen_limpieza = {
    "fecha_generacion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset_original": str(RUTA_DATASET_ORIGINAL),
    "dataset_limpio_v2": str(RUTA_DATASET_LIMPIO_V2),
    "dataset_limpio_v2_split": str(RUTA_DATASET_LIMPIO_V2_SPLIT),
    "registros_dataset_original": len(dataset_original),
    "registros_excluidos": len(indices_originales_excluir),
    "registros_dataset_limpio_v2": len(dataset_limpio_v2),
    "train_original": len(dataset_train),
    "test_original": len(dataset_test),
    "eval_original": len(dataset_eval),
    "train_limpio": len(dataset_train_limpio_v2),
    "test_limpio": len(dataset_test_limpio_v2),
    "eval_limpio": len(dataset_eval_limpio_v2),
    "seed_split_original": SEED
}

df_resumen_limpieza = pd.DataFrame([resumen_limpieza])

ruta_resumen_limpieza = RUTA_SALIDA_CUADERNO_05 / "resumen_limpieza_dataset_v2.csv"

df_resumen_limpieza.to_csv(
    ruta_resumen_limpieza,
    index=False,
    encoding="utf-8-sig"
)

print("Resumen final de limpieza guardado en:")
print(ruta_resumen_limpieza)

df_resumen_limpieza

Resumen final de limpieza guardado en:
C:\Lara\resultados\05_limpieza_dataset\resumen_limpieza_dataset_v2.csv


,fecha_generacion,dataset_original,dataset_limpio_v2,dataset_limpio_v2_split,registros_dataset_original,registros_excluidos,registros_dataset_limpio_v2,train_original,test_original,eval_original,train_limpio,test_limpio,eval_limpio,seed_split_original
0,2026-05-18 09:31:23,C:\Lara\datasets\lara_whisper_dataset,C:\Lara\datasets\lara_whisper_dataset_limpio_v2,C:\Lara\datasets\lara_whisper_dataset_limpio_v...,40842,23,40819,32673,6535,1634,32673,6512,1634,42


## Conclusiones

En este cuaderno se ha creado una versión limpia v2 del dataset mediante una limpieza parcial y conservadora.

La limpieza se ha basado en el análisis de errores realizado en el cuaderno 04, centrado en el conjunto de test del modelo mejorado.

A partir de ese análisis:

- Se cruzaron los resultados del test con el índice original del dataset.
- Se identificaron muestras sospechosas mediante métricas de error y posibles alucinaciones.
- Se revisaron manualmente los casos de mayor sospecha.
- Se marcaron como problemáticas únicamente las muestras confirmadas manualmente.
- Se eliminaron 23 muestras del dataset original completo.

El resultado final es un dataset limpio v2 con 40819 muestras.

#### Limitaciones de la limpieza realizada

No se ha revisado exhaustivamente todo el dataset.

Las muestras eliminadas proceden del conjunto de test, ya que eran las muestras sobre las que existían predicciones, métricas WER/CER y análisis detallado del modelo.

Por tanto, esta versión v2 debe entenderse como una primera mejora prudente del dataset, no como una limpieza completa definitiva.

Pueden seguir existiendo muestras problemáticas en los conjuntos de entrenamiento o evaluación que no hayan sido detectadas todavía.